# 15.4 Privacy in Machine Learning

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 15: Responsible AI & Future Trends

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-15-responsible-ai-and-future-trends/15.04_privacy-in-machine-learning.ipynb)

Companion code for Section 15.4 of the book. Run the cells in order from top to bottom.

**Example 15.4.1**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

# ── Differential Privacy - Gaussian Mechanism ─────────────────────
class DifferentialPrivacyMechanism:
    '''Implements basic differential privacy via Gaussian noise addition.'''

    def __init__(self, epsilon: float, delta: float = 1e-5):
        self.epsilon = epsilon
        self.delta   = delta

    def gaussian_noise_sigma(self, sensitivity: float) -> float:
        '''Compute noise std for (ε,δ)-DP via Gaussian mechanism.'''
        return sensitivity * np.sqrt(2 * np.log(1.25/self.delta)) / self.epsilon

    def privatise_mean(self, data: np.ndarray, sensitivity: float = None) -> float:
        '''Release differentially private mean.'''
        true_mean = data.mean()
        if sensitivity is None:
            # Sensitivity of mean = (max-min)/n
            sensitivity = (data.max()-data.min()) / len(data)
        sigma = self.gaussian_noise_sigma(sensitivity)
        return true_mean + np.random.normal(0, sigma)

    def privatise_histogram(self, data: np.ndarray, bins: int = 10) -> np.ndarray:
        '''Release differentially private histogram.'''
        counts, edges = np.histogram(data, bins=bins)
        # Sensitivity of histogram counts = 1 (changing one record changes at most 1 bin by 1)
        sigma = self.gaussian_noise_sigma(sensitivity=1.0)
        noisy_counts = counts + np.random.normal(0, sigma, len(counts))
        return np.maximum(0, noisy_counts), edges  # Clip negative counts

    def privatise_gradient(self, gradient: np.ndarray,
                             clip_norm: float = 1.0) -> np.ndarray:
        '''Clip and noise gradient (core of DP-SGD).'''
        # Clip by L2 norm (bounding sensitivity)
        norm = np.linalg.norm(gradient)
        if norm > clip_norm:
            gradient = gradient * clip_norm / norm
        # Add Gaussian noise
        sigma = self.gaussian_noise_sigma(sensitivity=clip_norm)
        return gradient + np.random.normal(0, sigma, gradient.shape)


# ── DP-SGD (Differentially Private Stochastic Gradient Descent) ───
class DPLogisticRegression:
    '''
    Logistic regression trained with differential privacy via DP-SGD.
    Each gradient update: clip per-sample gradients + add Gaussian noise.
    '''

    def __init__(self, epsilon=1.0, delta=1e-5, clip_norm=1.0,
                  lr=0.01, n_epochs=50, batch_size=32):
        self.epsilon    = epsilon
        self.delta      = delta
        self.clip_norm  = clip_norm
        self.lr         = lr
        self.n_epochs   = n_epochs
        self.batch_size = batch_size
        self.dp         = DifferentialPrivacyMechanism(epsilon, delta)
        self.weights_   = None
        self.bias_      = None

    def _sigmoid(self, z): return 1/(1+np.exp(-np.clip(z,-500,500)))

    def fit(self, X, y):
        n, d = X.shape
        self.weights_ = np.zeros(d)
        self.bias_    = 0.0

        for epoch in range(self.n_epochs):
            idx   = np.random.permutation(n)
            X_sh, y_sh = X[idx], y[idx]

            for i in range(0, n, self.batch_size):
                X_b = X_sh[i:i+self.batch_size]
                y_b = y_sh[i:i+self.batch_size]

                # Per-sample gradient computation (key to DP-SGD)
                p = self._sigmoid(X_b @ self.weights_ + self.bias_)
                err = p - y_b

                # Clip each per-sample gradient
                per_sample_grads = X_b * err.reshape(-1,1)
                clipped = []
                for g in per_sample_grads:
                    norm = np.linalg.norm(g)
                    clipped.append(g * min(1, self.clip_norm/norm) if norm > 0 else g)

                # Average + add noise
                mean_grad = np.mean(clipped, axis=0)
                priv_grad = self.dp.privatise_gradient(mean_grad, self.clip_norm)

                self.weights_ -= self.lr * priv_grad
                self.bias_    -= self.lr * np.mean(err)

        return self

    def predict_proba(self, X):
        p = self._sigmoid(X @ self.weights_ + self.bias_)
        return np.column_stack([1-p, p])

    def predict(self, X): return (self.predict_proba(X)[:,1] >= 0.5).astype(int)
    def score(self, X, y): return (self.predict(X)==y).mean()


# ── Compare: Standard LR vs DP-LR at different ε values ──────────
data = load_breast_cancer()
X, y = data.data, data.target
X_sc = StandardScaler().fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_sc, y, test_size=0.2, stratify=y, random_state=42)

print('=== Differential Privacy Trade-off: Privacy vs Accuracy ===')
print(f'{'Model':30s} {'Epsilon':>10} {'AUC':>8} {'Privacy'}')
print('='*60)

# Standard (no DP)
lr_std = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
auc_std = roc_auc_score(y_te, lr_std.predict_proba(X_te)[:,1])
print(f'{'Standard LogReg':30s} {'∞ (none)':>10} {auc_std:>8.4f}  No privacy')

# DP with different epsilon values
for epsilon in [10.0, 1.0, 0.5, 0.1]:
    dp_lr = DPLogisticRegression(epsilon=epsilon, clip_norm=1.0, lr=0.05, n_epochs=100)
    dp_lr.fit(X_tr, y_tr)
    auc_dp = roc_auc_score(y_te, dp_lr.predict_proba(X_te)[:,1])
    privacy = 'Weak' if epsilon>5 else 'Moderate' if epsilon>0.5 else 'Strong'
    cost = auc_std - auc_dp
    print(f'{'DP-LogReg':30s} {epsilon:>10.1f} {auc_dp:>8.4f}  {privacy} (cost: -{cost:.4f})')

# ── Membership Inference Attack demonstration ──────────────────────
print(f'\n=== Membership Inference Attack Simulation ===')
# Simple attack: if model confidence is high, sample was likely in training
train_proba = lr_std.predict_proba(X_tr)[:,1]
test_proba  = lr_std.predict_proba(X_te)[:,1]

# Compute max confidence (higher = more likely member)
train_conf  = np.maximum(train_proba, 1-train_proba)
test_conf   = np.maximum(test_proba, 1-test_proba)

# Predict membership: confident → likely training member
threshold   = np.percentile(np.concatenate([train_conf, test_conf]), 50)
true_labels = np.concatenate([np.ones(len(X_tr)), np.zeros(len(X_te))])
pred_member = np.concatenate([train_conf, test_conf]) > threshold

attack_acc  = (pred_member == true_labels).mean()
print(f'Attack accuracy: {attack_acc:.4f}')
print(f'Random guess:    0.5000')
print(f'Attack advantage:{attack_acc-0.5:.4f}  (higher = worse privacy)')
print(f'\nConclusion: Standard model leaks membership information.')
print(f'DP training reduces this attack advantage.')